In [4]:
import fiona
import geopandas as gpd

RAW_LAU_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\LAU_RG_01M_2024_4326.gpkg"
OUT_LAU_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_lau_clean_01M.gpkg"

TARGET_CNTR = {"FR", "BE", "IT", "LU"}
country_map = {"FR": "France", "BE": "Belgium", "IT": "Italy", "LU": "Luxembourg"}

# 1) list layers
layers = fiona.listlayers(RAW_LAU_GPKG)
print("LAU layers:", layers)
layer_name = layers[0]

# 2) read a small sample to see columns
sample = gpd.read_file(RAW_LAU_GPKG, layer=layer_name, rows=5)
print("Columns:", sample.columns.tolist())
print(sample.head(3))

# 3) auto-pick likely columns (works for GISCO LAU most of the time)
cols = sample.columns.tolist()

def pick_col(candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

CNTR_COL = pick_col(["CNTR_CODE", "CNTR_ID", "country_code", "COUNTRY", "CNTR"])
LAU_CODE_COL = pick_col(["LAU_ID", "LAU_CODE", "LAU", "GISCO_ID", "ID"])
LAU_NAME_COL = pick_col(["LAU_NAME", "NAME_LATN", "NAME", "LAU_LABEL"])

print("Auto-detected:")
print(" - country col:", CNTR_COL)
print(" - lau code col:", LAU_CODE_COL)
print(" - lau name col:", LAU_NAME_COL)

if any(v is None for v in [CNTR_COL, LAU_CODE_COL, LAU_NAME_COL]):
    raise KeyError("Could not auto-detect one of the required columns. Use the printed Columns list and set CNTR_COL/LAU_CODE_COL/LAU_NAME_COL manually.")

# 4) read full layer (only needed columns)
gdf = gpd.read_file(RAW_LAU_GPKG, layer=layer_name)[[CNTR_COL, LAU_CODE_COL, LAU_NAME_COL, "geometry"]].copy()
gdf = gdf.to_crs("EPSG:4326") if gdf.crs else gdf.set_crs("EPSG:4326")

# filter to your countries
gdf[CNTR_COL] = gdf[CNTR_COL].astype(str).str.strip()
gdf = gdf[gdf[CNTR_COL].isin(TARGET_CNTR)].copy()

# clean geometry once
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
try:
    gdf["geometry"] = gdf.geometry.make_valid()
except Exception:
    gdf["geometry"] = gdf.buffer(0)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

# standardize columns
gdf = gdf.rename(columns={
    CNTR_COL: "country_code",
    LAU_CODE_COL: "lau_code",
    LAU_NAME_COL: "lau_name",
})
gdf["country"] = gdf["country_code"].map(country_map)

gdf["lau_code"] = gdf["lau_code"].astype(str).str.strip()
gdf["lau_name"] = gdf["lau_name"].astype(str).str.strip()

lau_clean = gdf[["country", "country_code", "lau_code", "lau_name", "geometry"]].copy()

# save as a clean, small gpkg for your raster script
lau_clean.to_file(OUT_LAU_GPKG, layer="lau_units", driver="GPKG")

print("Saved:", OUT_LAU_GPKG)
print("Rows:", lau_clean.shape)
print("CRS:", lau_clean.crs)

LAU layers: ['LAU_RG_01M_2024_4326.gpkg']
Columns: ['GISCO_ID', 'CNTR_CODE', 'LAU_NAME', 'POP_2024', 'POP_DENS_2024', 'AREA_KM2', 'YEAR', 'geometry']
   GISCO_ID CNTR_CODE LAU_NAME  POP_2024  POP_DENS_2024    AREA_KM2  YEAR  \
0  AL_AL141        AL   Kurbin       0.0            0.0  269.178779  2024   
1  AL_AL142        AL    Lezhë       0.0            0.0  509.342849  2024   
2  AL_AL143        AL  Mirditë       0.0            0.0  870.249553  2024   

                                            geometry  
0  POLYGON ((19.55733 41.57767, 19.57182 41.59103...  
1  POLYGON ((19.8671 41.95292, 19.8707 41.94779, ...  
2  POLYGON ((19.8671 41.95292, 19.87091 41.9601, ...  
Auto-detected:
 - country col: CNTR_CODE
 - lau code col: GISCO_ID
 - lau name col: LAU_NAME
Saved: D:\M2_MoSEF\Acclim\DataCollection\data\europe_lau_clean_01M.gpkg
Rows: (43527, 5)
CRS: EPSG:4326


In [5]:
import os
import re
import glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG
# =========================
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

LAU_GPKG  = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_lau_clean_01M.gpkg"
LAU_LAYER = "lau_units"

COUNTRY_COL = "country"
COUNTRY_CODE_COL = "country_code"
LAU_CODE_COL = "lau_code"
LAU_NAME_COL = "lau_name"

OUT_DIR  = os.path.join(FLOODS_BASE, "_outputs_country_lau")
OUT_XLSX = os.path.join(OUT_DIR, "events_country_LAU_2015_2024.xlsx")

THRESHOLD_CM = 0.0
IGNORE_VALUES = {9999}
ALL_TOUCHED = True
ERROR_SAMPLE_LIMIT = 50

# =========================
# HELPERS
# =========================
def find_flood_roots(base_dir: str | Path, suffix: str = "_filtered") -> list[str]:
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(f"FLOODS_BASE not found: {base_dir}")
    roots = sorted(str(p) for p in base_dir.iterdir() if p.is_dir() and p.name.endswith(suffix))
    if not roots:
        raise FileNotFoundError(f"No '*{suffix}' folders found under: {base_dir}")
    return roots

def collect_tifs(roots: list[str]) -> list[str]:
    tifs = []
    for root in roots:
        tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
        tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))
    return sorted(set(tifs))

def infer_year_from_path(fp: str) -> int | None:
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    base = os.path.basename(fp)
    m = re.search(
        r"WD_MERGE_"
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2})"
        r"_duration_(?P<duration>\d+)_days"
        r"_cluster_(?P<cluster>\d+)"
        r"_A0_",
        base,
        flags=re.IGNORECASE
    )
    if not m:
        return None
    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": m.group("start"),
        "end_date": m.group("end"),
        "duration_days": int(m.group("duration")),
        "cluster_id": int(m.group("cluster")),
    }

def event_id_from_meta(meta: dict) -> str:
    return f"{meta['start_date']}---{meta['end_date']}_cluster_{meta['cluster_id']}"

def affected_and_max_cm(src, geom, threshold_cm: float):
    data, _ = mask(
        src, [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )
    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()
    if vals.size == 0:
        return False, None
    if IGNORE_VALUES:
        keep = np.ones(vals.shape, dtype=bool)
        for v in IGNORE_VALUES:
            keep &= (vals != v)
        vals = vals[keep]
        if vals.size == 0:
            return False, None
    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# =========================
# LOAD LAU (already clean + EPSG:4326)
# =========================
lau = gpd.read_file(LAU_GPKG, layer=LAU_LAYER)
lau = lau[lau.geometry.notna() & ~lau.geometry.is_empty].copy()

required = [COUNTRY_COL, COUNTRY_CODE_COL, LAU_CODE_COL, LAU_NAME_COL, "geometry"]
missing = [c for c in required if c not in lau.columns]
if missing:
    raise KeyError(f"Missing columns in LAU layer '{LAU_LAYER}': {missing}")

for c in [COUNTRY_COL, COUNTRY_CODE_COL, LAU_CODE_COL, LAU_NAME_COL]:
    lau[c] = lau[c].astype(str).str.strip()

lau = lau[required].copy()
lau_wgs84 = lau  # already EPSG:4326
sindex = lau_wgs84.sindex

# =========================
# DISCOVER FLOODS + GROUP FILES BY EVENT
# =========================
roots = find_flood_roots(FLOODS_BASE, suffix="_filtered")
print("Detected flood folders:")
for r in roots:
    print(" -", r)

tifs = collect_tifs(roots)
print("\nTotal TIFs found:", len(tifs))

event_to_files = defaultdict(list)
event_meta = {}
unparsed_files = []

for fp in tifs:
    meta = parse_event_from_name(fp)
    if meta is None:
        unparsed_files.append(fp)
        continue
    eid = event_id_from_meta(meta)
    event_to_files[eid].append(fp)
    if eid not in event_meta:
        m = meta.copy()
        m["event_id"] = eid
        event_meta[eid] = m

for eid in event_to_files:
    event_to_files[eid] = sorted(set(event_to_files[eid]))

print("Unparsed files:", len(unparsed_files))
print("Distinct events:", len(event_to_files))

# =========================
# MAIN LOOP
# =========================
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

long_rows = []
summary_rows = []

diag = Counter()
error_counts = Counter()
error_samples = []

for eid, files in tqdm(event_to_files.items(), desc="Processing events (LAU)"):
    diag["events_total"] += 1
    meta = event_meta[eid]

    lau_max = {}  # key=lau_code -> record with max_cm

    tested_total = 0
    overlap_skips_total = 0
    any_candidates = False
    tiles_processed = 0

    for fp in files:
        try:
            with rasterio.open(fp) as src:
                if src.crs is None:
                    diag["tiles_no_crs"] += 1
                    continue
                tiles_processed += 1

                try:
                    b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
                except Exception as e:
                    msg = f"transform_bounds_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue

                raster_bbox = box(*b)

                cand_idx = list(sindex.intersection(raster_bbox.bounds))
                if not cand_idx:
                    continue

                cands = lau_wgs84.iloc[cand_idx]
                cands = cands[cands.intersects(raster_bbox)]
                if cands.empty:
                    continue

                any_candidates = True

                cands_src = cands.to_crs(src.crs).copy()
                cands_src = cands_src[cands_src.geometry.notna() & ~cands_src.geometry.is_empty]
                if cands_src.empty:
                    continue

                raster_box_src = box(*src.bounds)

                for _, crow in cands_src.iterrows():
                    geom = crow.geometry
                    if geom is None or geom.is_empty:
                        continue

                    if not geom.intersects(raster_box_src):
                        overlap_skips_total += 1
                        continue

                    tested_total += 1

                    try:
                        ok, max_cm = affected_and_max_cm(src, geom, THRESHOLD_CM)
                    except Exception as e:
                        msg = f"mask_error: {repr(e)}"
                        error_counts[msg] += 1
                        if len(error_samples) < ERROR_SAMPLE_LIMIT:
                            error_samples.append((fp, msg))
                        continue

                    if not ok or max_cm is None:
                        continue

                    code = str(crow[LAU_CODE_COL]).strip()
                    name = str(crow[LAU_NAME_COL]).strip()
                    ctry = str(crow[COUNTRY_COL]).strip()
                    ctry_code = str(crow[COUNTRY_CODE_COL]).strip()

                    prev = lau_max.get(code)
                    if (prev is None) or (max_cm > prev["max_cm"]):
                        lau_max[code] = {
                            "country": ctry,
                            "country_code": ctry_code,
                            "lau_code": code,
                            "lau_name": name,
                            "max_cm": float(max_cm),
                        }

        except Exception as e:
            msg = f"tile_error: {repr(e)}"
            error_counts[msg] += 1
            if len(error_samples) < ERROR_SAMPLE_LIMIT:
                error_samples.append((fp, msg))
            continue

    if any_candidates:
        diag["events_with_candidates_bbox"] += 1
    if lau_max:
        diag["events_with_affected"] += 1

    touched_countries = set()
    for rec in lau_max.values():
        touched_countries.add(rec["country"])
        long_rows.append({
            "event_id": meta["event_id"],
            "year": meta["year"],
            "start_date": meta["start_date"],
            "end_date": meta["end_date"],
            "duration_days": meta["duration_days"],
            "cluster_id": meta["cluster_id"],
            "country": rec["country"],
            "country_code": rec["country_code"],
            "lau_code": rec["lau_code"],
            "lau_name": rec["lau_name"],
            "max_cm": rec["max_cm"],
        })

    summary_rows.append({
        "event_id": meta["event_id"],
        "year": meta["year"],
        "start_date": meta["start_date"],
        "end_date": meta["end_date"],
        "duration_days": meta["duration_days"],
        "cluster_id": meta["cluster_id"],
        "n_tiles": len(files),
        "n_tiles_processed": tiles_processed,
        "event_path": meta["event_path"],
        "n_countries": len(touched_countries),
        "countries": "; ".join(sorted(touched_countries)),
        "n_lau": len(lau_max),
        "n_candidates_tested": tested_total,
        "n_overlap_skipped": overlap_skips_total,
    })

df_lau_long = pd.DataFrame(long_rows)
df_summary = pd.DataFrame(summary_rows)

# Country roll-up
base_cols = ["event_id","year","start_date","end_date","duration_days","cluster_id","country","country_code"]
if not df_lau_long.empty:
    df_country = (
        df_lau_long.groupby(base_cols, dropna=False, as_index=False)
        .agg(n_lau=("lau_code","nunique"), max_cm=("max_cm","max"))
    )
else:
    df_country = pd.DataFrame(columns=base_cols + ["n_lau","max_cm"])

# Save Excel
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_country.to_excel(writer, sheet_name="events_country", index=False)
    df_lau_long.to_excel(writer, sheet_name="events_lau_long", index=False)
    df_summary.to_excel(writer, sheet_name="summary_event_lau", index=False)

print("\nSaved Excel:", OUT_XLSX)
print("Rows: events_country =", len(df_country), "| events_lau_long =", len(df_lau_long))
print("Diagnostics:", dict(diag))

Detected flood folders:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2015_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2016_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2017_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2018_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2019_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2020_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2022_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered

Total TIFs found: 1880
Unparsed files: 0
Distinct events: 1746


Processing events (LAU): 100%|████████████████████████████████████████████████████████████████████████████████████████| 1746/1746 [3:56:30<00:00,  8.13s/it]



Saved Excel: D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_country_lau\events_country_LAU_2015_2024.xlsx
Rows: events_country = 221 | events_lau_long = 5649
Diagnostics: {'events_total': 1746, 'events_with_candidates_bbox': 673, 'events_with_affected': 186}


In [6]:
import pandas as pd
import geopandas as gpd

# Inputs
XLSX_LAU = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_country_lau\events_country_LAU_2015_2024.xlsx"
LAU_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_lau_clean_01M.gpkg"
LAU_LAYER = "lau_units"

NUTS_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\europe_admin_clean_03M.gpkg"
REGION_LAYER = "regions_nuts2"
DEPT_LAYER = "departments_nuts3"

OUT_XLSX = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_country_lau\events_country_region_department_FROM_LAU.xlsx"

# 1) Load LAU hits
lau_long = pd.read_excel(XLSX_LAU, sheet_name="events_lau_long", dtype={"lau_code": str})
lau_long["lau_code"] = lau_long["lau_code"].astype(str).str.strip()
lau_long["max_cm"] = lau_long["max_cm"].astype(float)

base_cols = ["event_id","year","start_date","end_date","duration_days","cluster_id","country","country_code"]

# 2) Build LAU geometry mapping (lau_code -> point)
lau_geom = gpd.read_file(LAU_GPKG, layer=LAU_LAYER)[["lau_code","geometry"]].copy()
lau_geom["lau_code"] = lau_geom["lau_code"].astype(str).str.strip()
lau_geom = lau_geom.to_crs("EPSG:4326")

lau_pts = lau_geom.copy()
lau_pts["geometry"] = lau_pts.geometry.representative_point()

# 3) Load NUTS layers
depts = gpd.read_file(NUTS_GPKG, layer=DEPT_LAYER).to_crs("EPSG:4326")[["dept_code","dept_name","geometry"]]
regions = gpd.read_file(NUTS_GPKG, layer=REGION_LAYER).to_crs("EPSG:4326")[["region_code","region_name","geometry"]]

# 4) Spatial join: LAU -> NUTS3 and LAU -> NUTS2
lau2dept = gpd.sjoin(lau_pts, depts, predicate="intersects", how="left")[["lau_code","dept_code","dept_name"]]
lau2reg  = gpd.sjoin(lau_pts, regions, predicate="intersects", how="left")[["lau_code","region_code","region_name"]]

# 5) Merge mapping into LAU hits
df = lau_long.merge(lau2dept, on="lau_code", how="left").merge(lau2reg, on="lau_code", how="left")

# 6) Aggregate to dept/region/country
events_dept = (
    df.groupby(base_cols + ["dept_code","dept_name"], dropna=False, as_index=False)
      .agg(n_lau=("lau_code","nunique"), max_cm=("max_cm","max"))
)

events_region = (
    df.groupby(base_cols + ["region_code","region_name"], dropna=False, as_index=False)
      .agg(n_lau=("lau_code","nunique"), max_cm=("max_cm","max"))
)

events_country = (
    df.groupby(base_cols, dropna=False, as_index=False)
      .agg(
          n_lau=("lau_code","nunique"),
          n_departments=("dept_code","nunique"),
          n_regions=("region_code","nunique"),
          max_cm=("max_cm","max"),
      )
)

# 7) Save
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    events_country.to_excel(writer, "events_country", index=False)
    events_region.to_excel(writer, "events_regions_nuts2", index=False)
    events_dept.to_excel(writer, "events_departments_n3", index=False)
    df.to_excel(writer, "events_lau_long_enriched", index=False)

print("Saved:", OUT_XLSX)
print("Unique events affected (country table):", events_country["event_id"].nunique())

C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_24120\1979523123.py:64: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  events_country.to_excel(writer, "events_country", index=False)
C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_24120\1979523123.py:65: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  events_region.to_excel(writer, "events_regions_nuts2", index=False)
C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_24120\1979523123.py:66: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  events_dept.to_excel(writer, "events_departments_n3", index=False)
C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_24120\1979523123.py:67: FutureWarning: Starting with pandas version 3.0 all arguments of to_exc

Saved: D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_country_lau\events_country_region_department_FROM_LAU.xlsx
Unique events affected (country table): 186
